# Préparation des données ExpenseAI

Ce notebook nettoie et prépare les lignes de dépenses pour une future modélisation. Il ne réalise aucun entraînement et ne modifie jamais le fichier Excel source.

**Unité d'analyse :** une ligne de dépense. Une note de frais, identifiée par `Numéro (Dépense)`, peut contenir plusieurs lignes qui doivent être conservées.

## Périmètre et principes

- travailler sur une copie en mémoire du fichier source ;
- supprimer uniquement les doublons complets ;
- conserver les numéros répétés comme groupes de lignes ;
- retirer les informations provoquant une fuite de données ;
- conserver des variables lisibles, sans OneHotEncoding définitif ;
- exporter un CSV préparé dans `data/processed/`, dossier ignoré par Git.

## Import des bibliothèques

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px

warnings.filterwarnings("ignore", message="Workbook contains no default style")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
px.defaults.template = "plotly_white"

## 1. Chargement des données

Le classeur est uniquement lu. Toutes les transformations sont appliquées à `df`, une copie indépendante de `df_source`.

In [2]:
RACINE_PROJET = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FICHIER_SOURCE = RACINE_PROJET / "data" / "raw" / "expenses data-20260722102438.xlsx"

if not FICHIER_SOURCE.exists():
    raise FileNotFoundError(f"Fichier source introuvable : {FICHIER_SOURCE}")

df_source = pd.read_excel(FICHIER_SOURCE, sheet_name="data")
df = df_source.copy(deep=True)

print(f"Fichier lu sans modification : {FICHIER_SOURCE.name}")
print(f"Dimensions initiales : {df.shape[0]} lignes × {df.shape[1]} colonnes")

Fichier lu sans modification : expenses data-20260722102438.xlsx
Dimensions initiales : 7071 lignes × 14 colonnes


## 2. Suppression du véritable doublon complet

Deux lignes ne sont considérées comme dupliquées que si toutes leurs colonnes sont identiques. Un `Numéro (Dépense)` répété n'est pas un critère de suppression : il peut représenter plusieurs lignes distinctes d'une même note de frais.

In [3]:
nombre_avant = len(df)
nombre_doublons_complets = int(df.duplicated().sum())

# Seules les répétitions strictement identiques sur toutes les colonnes sont retirées.
df = df.drop_duplicates().copy()
nombre_apres = len(df)

rapport_dedoublonnage = pd.Series({
    "Nombre de lignes avant": nombre_avant,
    "Doublons complets supprimés": nombre_doublons_complets,
    "Nombre de lignes après": nombre_apres,
})
rapport_dedoublonnage.to_frame("Valeur")

,Valeur
Nombre de lignes avant,7071
Doublons complets supprimés,1
Nombre de lignes après,7070


**Résultat :** 7 071 lignes avant traitement, 1 doublon complet supprimé et 7 070 lignes conservées. Aucune ligne n'est supprimée en raison d'un numéro partagé.

## 3. Définition de la cible

La classe positive `1` représente une dépense refusée, événement minoritaire que le futur modèle devra détecter.

In [4]:
CORRESPONDANCE_CIBLE = {"Approuvée": 0, "Refusée": 1}
statuts_observes = set(df["Statut"].dropna().unique())
statuts_inattendus = statuts_observes - set(CORRESPONDANCE_CIBLE)

if statuts_inattendus or df["Statut"].isna().any():
    raise ValueError(
        f"Valeurs de Statut inattendues ou manquantes : {statuts_inattendus}"
    )

target = df["Statut"].map(CORRESPONDANCE_CIBLE).astype("int8")
print("Statuts observés :", sorted(statuts_observes))
print("Correspondance de la cible :", CORRESPONDANCE_CIBLE)
print(target.value_counts().sort_index().rename_axis("target").to_frame("Effectif"))

Statuts observés : ['Approuvée', 'Refusée']
Correspondance de la cible : {'Approuvée': 0, 'Refusée': 1}
        Effectif
target          
0           6956
1            114


## 4. Suppression du Data Leakage

Les colonnes connues après la décision, les identifiants techniques et les champs volontairement exclus de cette première version ne seront pas utilisés comme features.

In [5]:
COLONNE_GROUPE = "Numéro (Dépense)"
expense_group = df[COLONNE_GROUPE].astype("string").str.strip()
expense_group = expense_group.mask(expense_group.eq(""))
masque_groupe_absent = expense_group.isna()

# Chaque ligne sans numéro reçoit son propre groupe afin de ne pas créer une fausse note commune.
identifiants_techniques = pd.Series(
    [f"TECH_SANS_NUMERO_{index:06d}" for index in df.index[masque_groupe_absent]],
    index=df.index[masque_groupe_absent],
    dtype="string",
)
if set(identifiants_techniques) & set(expense_group.dropna()):
    raise ValueError("Collision entre un groupe technique et un numéro existant.")
expense_group.loc[masque_groupe_absent] = identifiants_techniques

assert expense_group.notna().all()
assert identifiants_techniques.nunique() == int(masque_groupe_absent.sum())
print(f"Numéros manquants remplacés par des groupes techniques uniques : {masque_groupe_absent.sum()}")
print(f"Nombre total de groupes : {expense_group.nunique()}")

Numéros manquants remplacés par des groupes techniques uniques : 10
Nombre total de groupes : 6253


In [6]:
COLONNES_FUITE = [
    "Date d'approbation",
    "Motif du refus",
    "Statut",
    "Nom de fichier (Justificatif)",
    "Numéro (Dépense)",
]
COLONNES_NON_RETENUES_V1 = [
    "Nom (Dépense)",             # Texte libre : traitement NLP et RGPD à étudier plus tard.
    "Nom (Projet)",              # Redondant avec Code projet dans cette première version.
    "Montant HT devise système", # Redondant avec le montant TTC retenu.
]

print("Colonnes exclues pour risque de fuite :", COLONNES_FUITE)
print("Colonnes non retenues dans la première version :", COLONNES_NON_RETENUES_V1)

Colonnes exclues pour risque de fuite : ["Date d'approbation", 'Motif du refus', 'Statut', 'Nom de fichier (Justificatif)', 'Numéro (Dépense)']
Colonnes non retenues dans la première version : ['Nom (Dépense)', 'Nom (Projet)', 'Montant HT devise système']


## 5. Traitement des dates

La date brute est convertie en variables temporelles interprétables. `jour_semaine` suit la convention pandas : 0 = lundi et 6 = dimanche. La colonne `Date` brute ne sera pas incluse dans les features finales.

In [7]:
date_depense = pd.to_datetime(df["Date"], errors="coerce")
dates_invalides = df["Date"].notna() & date_depense.isna()

if date_depense.isna().any():
    raise ValueError(
        f"La date de dépense contient {date_depense.isna().sum()} valeur(s) manquante(s) ou invalide(s)."
    )

variables_temporelles = pd.DataFrame(index=df.index)
variables_temporelles["annee"] = date_depense.dt.year.astype("int16")
variables_temporelles["mois"] = date_depense.dt.month.astype("int8")
variables_temporelles["jour"] = date_depense.dt.day.astype("int8")
variables_temporelles["jour_semaine"] = date_depense.dt.dayofweek.astype("int8")
variables_temporelles["trimestre"] = date_depense.dt.quarter.astype("int8")
variables_temporelles["est_weekend"] = date_depense.dt.dayofweek.isin([5, 6]).astype("int8")

print(f"Dates invalides détectées : {dates_invalides.sum()}")
variables_temporelles.head()

Dates invalides détectées : 0


,annee,mois,jour,jour_semaine,trimestre,est_weekend
0,2026,6,2,1,2,0
1,2026,5,25,0,2,0
2,2026,4,30,3,2,0
3,2026,5,26,1,2,0
4,2026,6,9,1,2,0


## 6. Traitement et rapport sur les montants

`Montant TTC devise système` est retenu comme montant principal. `Montant HT devise système` n'est pas utilisé simultanément afin de limiter la redondance. `Taux de taxe` est conservé. Les valeurs extrêmes sont signalées par la règle IQR (La règle IQR sert à repérer les valeurs potentiellement aberrantes, ce qu’on appelle des outliers), mais jamais supprimées automatiquement.

In [8]:
COLONNE_MONTANT = "Montant TTC devise système"
COLONNE_TAUX = "Taux de taxe"

montant_ttc = pd.to_numeric(df[COLONNE_MONTANT], errors="coerce")
taux_taxe = pd.to_numeric(df[COLONNE_TAUX], errors="coerce")

if montant_ttc.isna().any() or taux_taxe.isna().any():
    raise ValueError("Les variables numériques retenues contiennent des valeurs non convertibles.")

q1, q3 = montant_ttc.quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr
masque_extreme = (montant_ttc < borne_basse) | (montant_ttc > borne_haute)

rapport_montants = pd.Series({
    "Montants négatifs": int((montant_ttc < 0).sum()),
    "Montants nuls": int((montant_ttc == 0).sum()),
    "Valeurs extrêmes selon IQR": int(masque_extreme.sum()),
    "Borne IQR basse": round(borne_basse, 2),
    "Borne IQR haute": round(borne_haute, 2),
    "Minimum conservé": montant_ttc.min(),
    "Maximum conservé": montant_ttc.max(),
})
rapport_montants.to_frame("Valeur")

,Valeur
Montants négatifs,1.0
Montants nuls,2.0
Valeurs extrêmes selon IQR,629.0
Borne IQR basse,-99.5
Borne IQR haute,200.5
Minimum conservé,-17.7
Maximum conservé,3297.0


In [9]:
fig_montants = px.box(
    x=montant_ttc,
    points=False,
    title="Distribution du montant TTC — valeurs extrêmes conservées",
    labels={"x": "Montant TTC devise système"},
    color_discrete_sequence=["#2563EB"],
)
fig_montants

Le rapport identifie **1 montant négatif**, **2 montants nuls** et **629 valeurs hors bornes IQR**. Ces lignes sont toutes conservées : une valeur élevée peut être légitime selon le type de dépense. Une validation métier sera nécessaire avant toute correction future.

## 7. Traitement des valeurs manquantes

L'absence de code projet est une information en elle-même. Elle est représentée par la modalité explicite `SANS_PROJET`, sans inventer un projet.

In [10]:
valeurs_manquantes_avant = (
    df.isna().sum()
    .loc[lambda serie: serie > 0]
    .sort_values(ascending=False)
    .rename("Valeurs manquantes")
    .to_frame()
)
valeurs_manquantes_avant

,Valeurs manquantes
Motif du refus,6954
Code projet,4613
Nom (Projet),4613
Nom de fichier (Justificatif),833
Date d'approbation,114
Numéro (Dépense),10


## 8. Nettoyage des variables catégorielles

Les espaces en début et fin de chaîne sont supprimés, les chaînes vides deviennent manquantes et les variantes qui ne diffèrent que par la casse sont harmonisées selon l'écriture la plus fréquente. Le dataset actuel ne présente aucune collision de casse pour `Type` ou `Code projet`, mais la fonction sécurise les futurs exports.

In [11]:
def nettoyer_categorie(serie: pd.Series) -> pd.Series:
    """Nettoie une catégorie tout en conservant une écriture lisible."""
    valeurs = serie.astype("string").str.strip()
    valeurs = valeurs.mask(valeurs.eq(""))
    frequences = valeurs.value_counts()
    forme_canonique = {}
    for valeur, effectif in frequences.items():
        cle = valeur.casefold()
        if cle not in forme_canonique:
            forme_canonique[cle] = valeur
    return valeurs.map(lambda valeur: forme_canonique.get(valeur.casefold()) if pd.notna(valeur) else pd.NA)

type_depense = nettoyer_categorie(df["Type"])
code_projet = nettoyer_categorie(df["Code projet"]).fillna("SANS_PROJET")

if type_depense.isna().any():
    raise ValueError("La variable Type contient des valeurs manquantes après nettoyage.")

print(f"Modalités de Type : {type_depense.nunique()}")
print(f"Modalités de Code projet, SANS_PROJET inclus : {code_projet.nunique()}")
print(f"Lignes classées SANS_PROJET : {(code_projet == 'SANS_PROJET').sum()}")

Modalités de Type : 33
Modalités de Code projet, SANS_PROJET inclus : 195
Lignes classées SANS_PROJET : 4613


In [12]:
def convertir_facturable(serie: pd.Series) -> pd.Series:
    """Convertit les représentations usuelles de oui/non en variable binaire."""
    if pd.api.types.is_bool_dtype(serie):
        return serie.astype("int8")
    normalisee = serie.astype("string").str.strip().str.casefold()
    correspondance = {
        "true": 1, "vrai": 1, "oui": 1, "1": 1,
        "false": 0, "faux": 0, "non": 0, "0": 0,
    }
    resultat = normalisee.map(correspondance)
    if resultat.isna().any():
        raise ValueError("La variable Facturable contient une modalité inconnue.")
    return resultat.astype("int8")

facturable_binaire = convertir_facturable(df["Facturable"])
print(facturable_binaire.value_counts().sort_index().rename_axis("Facturable"))

Facturable
0    6392
1     678
Name: count, dtype: int64


## 9. Construction du dataset final

Les catégories restent lisibles. Leur futur encodage sera réalisé dans un pipeline scikit-learn ajusté uniquement sur les données d'entraînement.

In [13]:
colonnes_features = [
    "Type",
    "Montant TTC devise système",
    "Facturable",
    "Code projet",
    "Taux de taxe",
    "annee",
    "mois",
    "jour",
    "jour_semaine",
    "trimestre",
    "est_weekend",
]

dataset_prepare = pd.DataFrame(index=df.index)
dataset_prepare["expense_group"] = expense_group
dataset_prepare["Type"] = type_depense
dataset_prepare["Montant TTC devise système"] = montant_ttc.astype("float64")
dataset_prepare["Facturable"] = facturable_binaire
dataset_prepare["Code projet"] = code_projet
dataset_prepare["Taux de taxe"] = taux_taxe.astype("float64")
dataset_prepare = pd.concat([dataset_prepare, variables_temporelles], axis=1)
dataset_prepare["target"] = target

colonnes_interdites = set(COLONNES_FUITE + COLONNES_NON_RETENUES_V1 + ["Date"])
assert colonnes_interdites.isdisjoint(colonnes_features)
assert list(dataset_prepare.columns) == ["expense_group", *colonnes_features, "target"]
assert dataset_prepare["expense_group"].notna().all()
assert dataset_prepare["target"].isin([0, 1]).all()

In [14]:
comparaison_dimensions = pd.DataFrame({
    "Étape": ["Données originales", "Après suppression du doublon complet", "Dataset préparé"],
    "Lignes": [len(df_source), len(df), len(dataset_prepare)],
    "Colonnes": [df_source.shape[1], df.shape[1], dataset_prepare.shape[1]],
})
comparaison_dimensions

,Étape,Lignes,Colonnes
0,Données originales,7071,14
1,Après suppression du doublon complet,7070,14
2,Dataset préparé,7070,13


In [15]:
dataset_prepare.dtypes.rename("Type pandas").to_frame()

,Type pandas
expense_group,string[python]
Type,object
Montant TTC devise système,float64
Facturable,int8
Code projet,object
Taux de taxe,float64
annee,int16
mois,int8
jour,int8
jour_semaine,int8


In [16]:
valeurs_manquantes_restantes = dataset_prepare.isna().sum()
print("Valeurs manquantes restantes :", int(valeurs_manquantes_restantes.sum()))
valeurs_manquantes_restantes.rename("Nombre manquant").to_frame()

Valeurs manquantes restantes : 0


,Nombre manquant
expense_group,0
Type,0
Montant TTC devise système,0
Facturable,0
Code projet,0
Taux de taxe,0
annee,0
mois,0
jour,0
jour_semaine,0


In [17]:
repartition_target = pd.DataFrame({
    "Effectif": dataset_prepare["target"].value_counts().sort_index(),
    "Pourcentage (%)": (
        dataset_prepare["target"].value_counts(normalize=True).sort_index() * 100
    ).round(2),
})
repartition_target.index = ["Approuvée (0)", "Refusée (1)"]
repartition_target

,Effectif,Pourcentage (%)
Approuvée (0),6956,98.39
Refusée (1),114,1.61


### Premières lignes

Cette sortie contient des identifiants et reste strictement locale. Elle doit être effacée avant toute publication du notebook.

In [18]:
dataset_prepare.head()

,expense_group,Type,Montant TTC devise système,Facturable,Code projet,Taux de taxe,annee,mois,jour,jour_semaine,trimestre,est_weekend,target
0,NSAS260600032,Boissons sans alcool,11.62,0,RECRUTEMENT,0.1,2026,6,2,1,2,0,1
1,NSAS260500226,Petit équipement/petit matériel,89.99,0,SANS_PROJET,0.2,2026,5,25,0,2,0,1
2,NSAS260400448,Petit équipement/petit matériel,37.99,0,SANS_PROJET,0.2,2026,4,30,3,2,0,1
3,NSAS260500286,Event - MEET UP - Conférence,40.00,0,SANS_PROJET,0.0,2026,5,26,1,2,0,1
4,NSAS260600139,Abonnement Professionnel,19.99,0,FRONT_OFFICE,0.2,2026,6,9,1,2,0,1


In [19]:
print("Liste finale des features :")
for position, feature in enumerate(colonnes_features, start=1):
    print(f"{position:>2}. {feature}")

Liste finale des features :
 1. Type
 2. Montant TTC devise système
 3. Facturable
 4. Code projet
 5. Taux de taxe
 6. annee
 7. mois
 8. jour
 9. jour_semaine
10. trimestre
11. est_weekend


**Résultat final :** 7 070 lignes et 13 colonnes, dont 11 features, un identifiant de groupe et la cible. Aucune valeur manquante ne subsiste dans ce périmètre. Les 4 613 codes projet absents sont représentés par `SANS_PROJET` et les 10 numéros absents par 10 groupes techniques distincts.

## 10. Export du dataset préparé

Le CSV est exporté avec un encodage UTF-8. Il contient des données réelles et reste ignoré par Git. Aucun fichier Parquet n'est créé à ce stade afin de conserver uniquement les dépendances nécessaires.

In [20]:
FICHIER_SORTIE = RACINE_PROJET / "data" / "processed" / "expenses_clean.csv"
FICHIER_SORTIE.parent.mkdir(parents=True, exist_ok=True)
dataset_prepare.to_csv(FICHIER_SORTIE, index=False, encoding="utf-8")

print(f"Dataset exporté : {FICHIER_SORTIE}")
print(f"Nombre de lignes exportées : {len(dataset_prepare)}")

Dataset exporté : /Users/imanebenamar/Desktop/Projet final/ExpenseAI/data/processed/expenses_clean.csv
Nombre de lignes exportées : 7070


## 11. Préparation de la future modélisation

La cible reste fortement déséquilibrée : environ **98,39 % de lignes approuvées** contre **1,61 % de lignes refusées**. L'accuracy seule ne permettra donc pas d'évaluer correctement un futur modèle ; des métriques centrées sur la classe refusée seront nécessaires.

Le découpage train/test devra respecter `expense_group` : toutes les lignes portant le même `Numéro (Dépense)` devront idéalement rester dans le même jeu. Les identifiants techniques créés pour les numéros absents sont uniques par ligne et ne créent pas de regroupement artificiel.

Tout éventuel rééquilibrage, par `class_weight`, `RandomUnderSampler`, SMOTE ou une autre méthode, devra être appliqué **uniquement aux données d'entraînement**, après la séparation, afin d'éviter toute contamination du jeu de test. Aucune de ces méthodes n'est appliquée dans ce notebook.

Le preprocessing Machine Learning définitif — imputation éventuelle, encodage catégoriel et transformations numériques — sera encapsulé dans un `Pipeline` scikit-learn ajusté uniquement sur le jeu d'entraînement. Cette organisation limitera le data leakage et garantira la reproductibilité.

**Aucun modèle, SMOTE, GridSearchCV, SHAP, traitement PostgreSQL ou changement Streamlit n'est réalisé à cette étape.**